# BLP + ML Instruments — Walkthrough

Pipeline completo: dados sintéticos → candidatos → seleção ML → diagnóstico → BLP

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simulate_data import simulate_blp_data
from instruments import (
    build_instrument_candidates,
    select_instruments_lasso,
    select_instruments_rf,
    select_instruments_combined,
)
from diagnostics import first_stage, sargan_hansen_test, plot_first_stage, plot_instrument_importance

## 1. Dados sintéticos

In [ ]:
df = simulate_blp_data(T=50, J=10, seed=42)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Verificar endogeneidade: correlação entre preço e xi (qualidade não observada)
print(f'Correlação preço × xi: {df["price"].corr(df["xi"]):.3f}')
print('Esperado: correlação positiva alta (endogeneidade por construção)')

## 2. Candidatos a instrumento

In [ ]:
CHAR_COLS = ['x1', 'x2', 'x3']
candidates = build_instrument_candidates(df, CHAR_COLS)
print(f'Candidatos: {candidates.shape[1]} variáveis')
candidates.head()

## 3. Seleção via Lasso

In [ ]:
Z_lasso, lasso_cols = select_instruments_lasso(
    candidates, df['price'], X_controls=df[CHAR_COLS]
)
print(f'\nSelecionados: {len(lasso_cols)}')

## 4. Seleção via Random Forest

In [ ]:
Z_rf, importances = select_instruments_rf(candidates, df['price'], threshold=0.01)
plot_instrument_importance(importances, top_n=15)
plt.show()

## 5. Seleção combinada (interseção)

In [ ]:
Z_combined = select_instruments_combined(candidates, df['price'], df[CHAR_COLS])

## 6. Diagnóstico

In [ ]:
fs = first_stage(df['price'], Z_combined, df[CHAR_COLS])
fig = plot_first_stage(df['price'], Z_combined, df[CHAR_COLS])
plt.show()

In [ ]:
outcome = np.log(df['shares'] / df['outside_share'])
sargan = sargan_hansen_test(df['price'], outcome, Z_combined, df[CHAR_COLS])

## 7. Estimação BLP

Requer `pip install pyblp`

In [ ]:
try:
    import pyblp

    df_blp = df.copy()
    for i, col in enumerate(Z_combined.columns):
        df_blp[f'demand_instruments{i}'] = Z_combined[col].values

    problem = pyblp.Problem(
        product_formulations=(
            pyblp.Formulation('1 + x1 + x2 + x3 + price'),
            pyblp.Formulation('0 + x1 + x2 + x3'),
        ),
        product_data=df_blp,
    )

    print(problem)

    results = problem.solve(
        sigma=np.diag([0.5, 0.5, 0.5]),
        optimization=pyblp.Optimization('bfgs'),
    )

    print(results)

except ImportError:
    print('pyblp não instalado: pip install pyblp')